# Lección 11.5 — Seq2seq y el cuello de botella

> **Módulo 11 · Secuencias y lenguaje** · Nivel 4 — Avanzado
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 11.2 (*embeddings*), Lección 11.3 (RNN y generación letra a letra), Lección 11.4 (GRU)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar la arquitectura **codificador–decodificador** (*encoder–decoder*, seq2seq) con una analogía.
- Distinguir cómo se **entrena** (*teacher forcing*) y cómo se **usa** (generación paso a paso) un seq2seq.
- Entrenar un seq2seq que **invierte** secuencias y medir dónde falla.
- Explicar el **cuello de botella** del vector de contexto y por qué motivó la **atención** (Módulo 12).

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación

In [ ]:
import time
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

plt.rcParams["figure.figsize"] = (9, 3.5)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("✅ Todo listo · dispositivo:", device)

---
## 1. 📖 La intuición: el intérprete con un solo post-it

Imagina a un intérprete en una conferencia con una regla extraña:
1. Escucha **todo** el discurso en español sin interrumpir.
2. Mientras escucha, solo puede tomar notas en **un post-it** de tamaño fijo.
3. Cuando el orador termina, guarda el discurso **solo** en su cabeza a través del post-it y empieza a hablar en inglés, **palabra por palabra**, sin volver a oír el original.

```
  "Me gusta mucho el café"                         "I really like coffee"
   │   │    │     │   │                              ▲    ▲     ▲     ▲
   ▼   ▼    ▼     ▼   ▼                              │    │     │     │
 [🎧]→[🎧]→[🎧]→[🎧]→[🎧] ══► 📝 post-it ══►  [🗣️]→[🗣️]→[🗣️]→[🗣️]
 └──────── CODIFICADOR ────────┘ (contexto)   └──── DECODIFICADOR ────┘
```

Esa es la arquitectura **seq2seq** (*sequence to sequence*, de secuencia a secuencia), con la que Google rehízo su traductor en 2016:
- El **codificador** (*encoder*) es una RNN/LSTM/GRU que lee la entrada y deja su memoria final: el **vector de contexto** (el post-it).
- El **decodificador** (*decoder*) es otra RNN que **arranca** con ese post-it y genera la salida token por token, como la RNN de nombres de la Lección 11.3.

La gran ventaja: **la entrada y la salida pueden tener longitudes distintas** ("Me gusta mucho el café" tiene 5 palabras; "I really like coffee", 4).

### ✋ Ejercicio 1 — ¿Qué tareas son seq2seq?
¿En cuáles la entrada **y** la salida son secuencias (de longitud posiblemente distinta)?
- a) Traducir del español al inglés
- b) Decidir si una reseña es positiva o negativa
- c) Resumir una noticia
- d) Pasar una grabación de voz a texto
- e) Predecir las ventas de mañana (un solo número)

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

**a), c) y d)** son seq2seq: entra una secuencia y sale otra de longitud distinta (palabras → palabras, noticia larga → resumen corto, audio → letras).

- **b)** es secuencia → **una** etiqueta (clasificación): basta con el codificador.
- **e)** es secuencia → **un** número: lo que hicimos en la 11.3 con la señal. (Si quisieras predecir los **próximos 7 días**, sí sería seq2seq.)
</details>

---
## 2. 📖 El concepto: dos fases y un token de arranque

El decodificador genera **un token a la vez**, y cada token que genera es la **entrada** del paso siguiente. Para el primer paso le damos un token especial `<inicio>`; en tareas de longitud variable, el decodificador también aprende a emitir `<fin>` para decir "ya terminé" (como el `.` de los nombres en la 11.3).

```
Decodificador:   <inicio> ──► "I" ──► "really" ──► "like" ──► "coffee" ──► <fin>
                     │        ▲ │       ▲ │          ▲ │         ▲ │
                     └────────┘ └───────┘ └──────────┘ └─────────┘ └──► (cada salida es la siguiente entrada)
```

### 📖 Entrenar vs. usar: *teacher forcing*
- **Al entrenar**, ¿qué le damos como entrada en el paso 3: la palabra que **generó** el decodificador en el paso 2 (que al principio será basura) o la palabra **correcta**? Se le da la **correcta**. Eso se llama ***teacher forcing*** ("forzar con el maestro"): como un profesor que, si el alumno se equivoca en una palabra, le dice la buena para que siga practicando la siguiente. Entrena mucho más rápido y se puede calcular todo de una vez.
- **Al usar** el modelo (inferencia) no hay maestro: el decodificador se alimenta de **sus propias** predicciones.

### ✋ Ejercicio 2
Queremos que el modelo traduzca "hola mundo" → "hello world". Escribe qué recibe el decodificador **como entrada** en cada paso:
1. Durante el **entrenamiento** (con *teacher forcing*).
2. Durante el **uso**, si en el paso 1 el modelo se equivoca y dice "hi".

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. **Entrenamiento:** `<inicio>` → `hello` → `world`. Recibe siempre las palabras **correctas**, aunque se haya equivocado al predecirlas; su objetivo en cada paso es `hello`, `world`, `<fin>`.
2. **Uso:** `<inicio>` → `hi` → (lo que haya generado después de "hi")… Un error al principio **se arrastra**: el resto se genera a partir de "hi".

Esa diferencia entre entrenar con respuestas perfectas y usar el modelo con sus propios errores tiene nombre: ***exposure bias*** (sesgo de exposición). Es uno de los errores comunes de la Sección 6.
</details>

---
## 3. 📖 Manos a la obra: un seq2seq que **invierte** secuencias

Para ver el mecanismo sin la complicación de un idioma real, usaremos una tarea de juguete:

```
entrada:  3 1 4 1 5   →   salida:  5 1 4 1 3
```

Parece trivial, pero para el modelo es un reto puro de **memoria**: tiene que meter **toda** la secuencia en el post-it y luego sacarla al revés. No puede "copiar" mirando la entrada.

Entrenaremos con secuencias de **3 a 12 dígitos**.

In [ ]:
INICIO = 10                     # token especial: los dígitos son 0–9, así que <inicio> es el 10
VOCAB = 11

def lote_inversion(n, largo, generador):
    x = torch.randint(0, 10, (n, largo), generator=generador)
    return x, x.flip(1)         # la salida correcta es la entrada al revés

gen = torch.Generator().manual_seed(0)
x, y = lote_inversion(3, 5, gen)
for a, b in zip(x.tolist(), y.tolist()):
    print(a, "→", b)

### 📖 El modelo
- **Codificador:** *embedding* de cada dígito (Lección 11.2) → GRU (Lección 11.4). Nos quedamos **solo** con su memoria final: el vector de contexto.
- **Decodificador:** otra GRU que **arranca** con ese contexto, recibe el dígito anterior y predice el siguiente.

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, oculto=64, dim_emb=16):
        super().__init__()
        self.emb = nn.Embedding(VOCAB, dim_emb)
        self.codificador = nn.GRU(dim_emb, oculto, batch_first=True)
        self.decodificador = nn.GRU(dim_emb, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 10)

    def forward(self, x, y):                           # ENTRENAMIENTO (con teacher forcing)
        _, contexto = self.codificador(self.emb(x))    # 📝 el post-it: un vector de `oculto` números
        inicio = torch.full((len(y), 1), INICIO)
        entrada_dec = torch.cat([inicio, y[:, :-1]], dim=1)   # <inicio> + la respuesta correcta corrida
        salidas, _ = self.decodificador(self.emb(entrada_dec), contexto)
        return self.salida(salidas)

    @torch.no_grad()
    def invertir(self, x):                             # USO (se alimenta de sus propias predicciones)
        _, h = self.codificador(self.emb(x))
        token = torch.full((len(x), 1), INICIO)
        resultado = []
        for _ in range(x.shape[1]):
            salida, h = self.decodificador(self.emb(token), h)
            token = self.salida(salida).argmax(-1)     # el dígito más probable…
            resultado.append(token)                    # …se guarda y será la próxima entrada
        return torch.cat(resultado, dim=1)

### 🔮 Ejercicio 3 — Antes de entrenar
El modelo está recién creado (pesos al azar). Si le pedimos invertir secuencias de 5 dígitos, ¿qué porcentaje de dígitos crees que acertará?

✍️ *Mi predicción:*

In [ ]:
torch.manual_seed(0)
modelo = Seq2Seq(oculto=64)
x, y = lote_inversion(500, 5, torch.Generator().manual_seed(1))
print(f"Dígitos acertados sin entrenar: {(modelo.invertir(x) == y).float().mean():.0%}")

<details><summary><b>👀 Ver respuesta</b></summary>

Alrededor del **10 %**: con 10 dígitos posibles, adivinar al azar acierta 1 de cada 10. (Puede salir un poco más o menos según la semilla; un modelo sin entrenar suele repetir mucho el mismo dígito.)
</details>

### ▶️ Entrenamos
En cada paso elegimos una longitud al azar entre 3 y 12. ▶️ Tarda ≈10–30 s.

In [ ]:
def entrenar_seq2seq(oculto=64, pasos=1500, semilla=0):
    torch.manual_seed(semilla); gen = torch.Generator().manual_seed(semilla)
    modelo = Seq2Seq(oculto)
    opt = torch.optim.Adam(modelo.parameters(), lr=0.005)
    inicio = time.time()
    for paso in range(pasos):
        largo = int(torch.randint(3, 13, (1,), generator=gen))
        x, y = lote_inversion(64, largo, gen)
        perdida = F.cross_entropy(modelo(x, y).reshape(-1, 10), y.reshape(-1))
        opt.zero_grad(); perdida.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
    print(f"oculto={oculto}: entrenado en {time.time() - inicio:.0f} s · pérdida final {perdida.item():.3f}")
    return modelo

modelo = entrenar_seq2seq(oculto=64)

for secuencia in [[3, 1, 4], [3, 1, 4, 1, 5], [2, 7, 1, 8, 2, 8, 1, 8, 2, 8]]:
    x = torch.tensor([secuencia])
    print(f"{secuencia} → {modelo.invertir(x)[0].tolist()}   (correcto: {secuencia[::-1]})")

### ✋ Ejercicio 4 — ¿Dónde se equivoca?
Mira la secuencia de 10 dígitos. Si hay errores, ¿están al **principio** o al **final** de la salida? ¿A qué parte de la **entrada** corresponden?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Las secuencias cortas salen perfectas. En la de 10 dígitos hay **varios errores** (en nuestra ejecución, la mitad de los dígitos), pero el **primer** dígito de la salida, que es el **último** que leyó el codificador, sale bien. Con un solo ejemplo es difícil ver un patrón claro: en la Sección 4 lo mediremos con 500 secuencias y verás que los errores se concentran en lo que el codificador leyó **primero**, lo que tuvo que sobrevivir más pasos dentro del post-it.
</details>

---
## 4. 📖 Qué pasa si… la secuencia se alarga: el cuello de botella

Midamos con 500 secuencias de cada longitud:
- **Exacta:** porcentaje de secuencias invertidas **sin ningún error**.
- **Por dígito:** porcentaje de dígitos correctos.

### 🔮 Ejercicio 5 — Predice la curva
El modelo entrenó con longitudes de 3 a 12. ¿Cómo crees que caerá la exactitud **exacta** al pasar de 5 a 8, 10 y 12 dígitos? ¿Y qué pasará con 15 y 20 dígitos, que **nunca** vio?

✍️ *Mis predicciones:*

In [ ]:
def evaluar(modelo, largos, n=500, semilla=1):
    gen = torch.Generator().manual_seed(semilla)
    resultados = {}
    for largo in largos:
        x, y = lote_inversion(n, largo, gen)
        aciertos = modelo.invertir(x) == y
        resultados[largo] = aciertos
        print(f"largo {largo:>2}: exacta {aciertos.all(1).float().mean():>4.0%} · por dígito {aciertos.float().mean():>4.0%}")
    return resultados

resultados = evaluar(modelo, [3, 5, 8, 10, 12, 15, 20])

<details><summary><b>👀 Ver respuesta</b></summary>

- 3 y 5 dígitos: **100 %** exacta.
- 8 dígitos: ≈**75 %** · 10: ≈**30 %** · 12: ≈**5 %** de secuencias perfectas. Por dígito, de ≈92 % a ≈60 %.
- 15 y 20 (nunca vistas): **0 %** exacta y menos de la mitad de los dígitos bien.

Aunque el modelo entrenó con secuencias de hasta 12, en las largas **casi nunca** acierta completa. La razón está en el post-it… 👇
</details>

### ▶️ ¿Qué posiciones fallan?
Para las secuencias de 12 dígitos, veamos el acierto en **cada posición de la salida**. Recuerda: la posición 1 de la salida es el **último** dígito que leyó el codificador, y la 12, el **primero**.

In [ ]:
por_posicion = resultados[12].float().mean(0).numpy()
plt.bar(range(1, 13), por_posicion)
plt.axhline(0.1, color="gray", ls="--", label="adivinar al azar")
plt.xticks(range(1, 13)); plt.ylim(0, 1.05)
plt.xlabel("posición en la SALIDA  (1 = último dígito leído · 12 = primer dígito leído)")
plt.ylabel("acierto"); plt.title("Invertir 12 dígitos: ¿qué se olvida?"); plt.legend(); plt.show()
print([f"{p:.0%}" for p in por_posicion])

### ✋ Ejercicio 6 — Diagnostica
1. ¿Qué dígitos se recuerdan mejor y cuáles peor?
2. Explica el resultado con la analogía del intérprete y el post-it.

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. La posición 1 de la salida (el **último** dígito leído) se acierta casi siempre (≈97 %); a partir de ahí el acierto cae poco a poco, hasta ≈**35 %** en la posición 12 (el **primer** dígito leído).
2. El codificador tiene que meter **12 dígitos** en un vector de **64 números**, y lo leído al principio se va "pisando" con lo que llega después (aunque sea una GRU, Lección 11.4). El decodificador **solo** tiene ese post-it: no puede volver a mirar la entrada.

Esto es el **cuello de botella** (*bottleneck*) de seq2seq: **toda** la información de la entrada, sea de 3 o de 300 palabras, tiene que pasar por **un único vector de tamaño fijo**. Es como resumir un libro entero en un post-it y luego intentar recitarlo.
</details>

### ✏️ Ejercicio 7 — ¿Y con un post-it más chico o más grande?
El tamaño del post-it es `oculto`. 🔮 Si lo bajamos de 64 a **16**, ¿qué le pasará a la exactitud con 8 y 10 dígitos? Y si lo subiéramos a 128, ¿desaparecería el problema?

▶️ Con `oculto=16` tarda ≈10 s; con 128, ≈30–40 s.

In [ ]:
OCULTO = 16        # ✏️ prueba 16, 32, 128
modelo_otro = entrenar_seq2seq(oculto=OCULTO)
_ = evaluar(modelo_otro, [5, 8, 10, 12])

<details><summary><b>👀 Ver respuesta</b></summary>

- Con **16**: ya falla con 5 dígitos (≈85–90 % exacta) y con 8 o más casi nunca acierta completo (≈10 % con 8, 0 % con 10). El post-it es demasiado chico.
- Con **128** (si lo pruebas): mejora mucho (≈90 % exacta con 8, ≈60 % con 10, ≈30 % con 12), pero **no** resuelve el problema: las secuencias más largas siguen fallando y ahora el modelo tiene 4 veces más parámetros en su GRU.

Agrandar el post-it solo **mueve** el límite. La solución de fondo fue otra: **dejar de depender de un único post-it**. 👇
</details>

### 📖 La pista que lleva a la atención
Mira de nuevo el codificador: `salidas, contexto = self.codificador(...)`. Nosotros **tiramos** `salidas` y nos quedamos solo con `contexto` (la memoria del último paso). Pero `salidas` contiene **la memoria en cada paso**: ¡una nota por cada dígito leído!

In [ ]:
x, _ = lote_inversion(1, 12, torch.Generator().manual_seed(5))
salidas, contexto = modelo.codificador(modelo.emb(x))
print("Notas del codificador, una por paso:", tuple(salidas.shape), "← ¡las estamos tirando!")
print("El post-it (solo el último paso)   :", tuple(contexto.shape))

### ✋ Ejercicio 8 — Inventa la atención
Al generar la **posición 12** de la salida (el primer dígito de la entrada), ¿qué nota del codificador le serviría más al decodificador? ¿Qué le propondrías hacer, en lugar de depender solo del post-it?

✍️ *Mi propuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Le serviría la nota del **paso 1** del codificador, que se escribió justo al leer ese dígito. Propuesta: que en **cada** paso el decodificador pueda **mirar todas las notas** del codificador y decidir **cuáles le importan ahora** (para la posición 12, la del paso 1; para la posición 1, la del paso 12).

¡Acabas de inventar la **atención** (Bahdanau et al., 2014)! En lugar de un solo post-it, el decodificador consulta una **biblioteca de notas** y elige en qué fijarse en cada momento. Con atención, la traducción automática mejoró enormemente en frases largas, y tres años después, el artículo *"Attention Is All You Need"* (2017) quitó por completo las redes recurrentes: así nacieron los **Transformers**. Es el tema de la Lección 12.1.
</details>

---
## 5. 📖 Hiperparámetros y perillas

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `hidden_size` (tamaño del contexto) | Cuánta información cabe en el post-it | Secuencias más largas; más parámetros | Cuello de botella más estrecho | 256–1024 (en traducción real) |
| `num_layers` | Capas de codificador y decodificador | Más capacidad | Más simple | 2–4 (traducción) |
| Proporción de *teacher forcing* | % de pasos en que el decodificador recibe la respuesta correcta al entrenar | Entrena rápido pero sufre más *exposure bias* | Más robusto a sus propios errores, más lento | 1.0, bajándola poco a poco (0.5) |
| Ancho del haz (*beam size*) | Cuántas candidatas parciales se mantienen al generar | Mejores secuencias globales; más lento | 1 = voraz (*greedy*): la mejor en cada paso | 4–10 (traducción) |
| Longitud máxima de salida | Tope de tokens a generar | Puede "divagar" si no emite `<fin>` | Corta respuestas | Según la tarea |

> 💡 **Búsqueda en haz (*beam search*):** nuestro `invertir` elige en cada paso el dígito más probable (**búsqueda voraz**). Pero la mejor secuencia completa no siempre empieza por la mejor primera palabra. La búsqueda en haz mantiene las *k* mejores frases parciales a la vez y al final elige la de mayor probabilidad total. Lo retomarás con las perillas de generación de los LLM (Lección 13.6).

### ✋ Ejercicio 9
Tu traductor funciona bien en validación (calculada con *teacher forcing*), pero al usarlo de verdad produce frases que se "descarrilan" tras un error. ¿Qué perilla tocarías y por qué?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Es ***exposure bias***: al entrenar siempre recibió las palabras correctas, así que nunca aprendió a recuperarse de sus propios errores. Tocaría la **proporción de *teacher forcing***: bajarla poco a poco durante el entrenamiento (a veces le damos su propia predicción), para que practique en las condiciones reales. También usaría **búsqueda en haz**, que evita comprometerse con una mala primera palabra. Y mediría la validación **generando**, sin maestro, como en el uso real.
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| Frases cortas perfectas, frases largas pésimas | Cuello de botella del vector de contexto | Más `hidden_size`; mejor: **atención** (M12) |
| Buen error de validación pero mala generación real | Validaste con *teacher forcing* (*exposure bias*) | Validar generando; bajar la proporción de *teacher forcing* |
| El modelo nunca termina o repite en bucle | No aprendió a emitir `<fin>`, o búsqueda voraz | Longitud máxima, *beam search*, penalizar repeticiones (13.6) |
| Falla con secuencias más largas que las de entrenamiento | No generaliza fuera del rango visto | Entrenar con el rango de longitudes que verá en uso |
| El inicio de la entrada se "olvida" | El contexto pierde lo leído al principio | Invertir la entrada (truco de Sutskever, 2014), LSTM/GRU bidireccional, atención |
| Errores de formas en el decodificador | Confundir el estado `h` (1, lote, oculto) con las salidas (lote, pasos, oculto) | Imprime `.shape` en cada paso |

### 🔮 Ejercicio 10 — El truco de Sutskever
En 2014, Sutskever y sus coautores descubrieron que su traductor mejoraba mucho si **invertían la frase de entrada** (pero no la de salida). Con lo que viste en el Ejercicio 6, ¿por qué crees que funcionaba?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

En una traducción, la **primera** palabra de la salida suele corresponder a la **primera** de la entrada. Sin invertir, esa palabra se leyó **hace mucho** cuando el decodificador empieza: la más olvidada. Al invertir la entrada, la primera palabra de la entrada se lee **al final**, justo antes de empezar a generar: queda "fresca" en el post-it. Y como un buen comienzo ayuda a todo lo demás (la salida se alimenta de sí misma), la traducción completa mejora.

Es el mismo efecto que viste en el Ejercicio 6, aprovechado a favor. Un truco ingenioso para un problema de fondo (el cuello de botella) que la atención resolvió de verdad.
</details>

---
## 7. 📋 Ficha del modelo

| | Seq2seq (codificador–decodificador recurrente) |
|---|---|
| **Tipo** | Secuencia → secuencia (longitudes distintas): traducción, resumen, voz a texto, pronóstico de varios pasos |
| **Intuición** | Un intérprete que escucha todo, lo resume en un post-it y lo recita en otro idioma |
| **Cómo aprende** | Codificador + decodificador entrenados juntos con *teacher forcing* y entropía cruzada por token |
| **Hiperparámetros clave** | `hidden_size`, `num_layers`, proporción de *teacher forcing*, ancho del haz |
| **✅ Fortalezas** | Entrada y salida de cualquier longitud · Un solo modelo de punta a punta · Base de la traducción neuronal |
| **❌ Debilidades** | **Cuello de botella** del contexto · Pierde calidad con secuencias largas · *Exposure bias* · Lento (secuencial) |
| **Úsalo cuando…** | Secuencias cortas, poco cómputo, o como paso para entender Transformers |
| **Evítalo cuando…** | Las entradas son largas: usa seq2seq **con atención** o Transformers (M12) |

## 8. 🧠 Resumen en 5 puntos
1. **Seq2seq** = un **codificador** que lee la entrada y la resume en un **vector de contexto** + un **decodificador** que genera la salida token por token a partir de él.
2. Se **entrena** con *teacher forcing* (el decodificador recibe las respuestas correctas) y se **usa** alimentándose de sus propias predicciones, desde un token `<inicio>` hasta `<fin>`.
3. Su gran ventaja: entrada y salida de **longitudes distintas** (traducir, resumir, transcribir).
4. Su gran debilidad: el **cuello de botella**. Toda la entrada pasa por un vector de tamaño fijo, así que con secuencias largas olvida (sobre todo lo que leyó primero). Agrandar el vector solo mueve el límite.
5. La solución fue dejar que el decodificador **mire todas las notas** del codificador y elija cuáles le importan: la **atención**, punto de partida del Módulo 12.

---
## 9. 🏋️ Ejercicios finales

### 9.1 Repaso rápido (opción múltiple)

**1.** En seq2seq, el vector de contexto es…
- a) La primera palabra de la salida · b) La memoria final del codificador, que resume toda la entrada · c) El vocabulario · d) La pérdida

**2.** Durante el entrenamiento con *teacher forcing*, el decodificador recibe como entrada…
- a) Sus propias predicciones · b) Las palabras correctas del paso anterior · c) Solo el contexto · d) Ruido

**3.** Tu seq2seq invierte bien secuencias de 5 dígitos pero falla con las de 12. ¿Qué cambio **ataca la causa de fondo**?
- a) Más épocas · b) Añadir atención para que el decodificador mire todas las notas del codificador · c) Bajar la tasa de aprendizaje · d) Usar menos dígitos

**4.** ¿Cuál de estas tareas **no** necesita un decodificador?
- a) Traducir · b) Resumir · c) Clasificar el sentimiento de una reseña · d) Transcribir audio

**5.** La búsqueda en haz (*beam search*) sirve para…
- a) Entrenar más rápido · b) Generar secuencias de mayor probabilidad total que la búsqueda voraz · c) Reducir el vocabulario · d) Evitar el sobreajuste

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** El post-it.
2. **b)** Las respuestas correctas, aunque se haya equivocado.
3. **b)** El problema es el cuello de botella; más épocas o tasa de aprendizaje no lo resuelven.
4. **c)** Secuencia → una etiqueta: basta con el codificador y una capa de clasificación.
5. **b)** Mantiene varias candidatas en lugar de comprometerse con la mejor de cada paso.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Explica con la analogía del intérprete por qué un seq2seq sin atención traduce bien "Buenos días" pero mal un párrafo de 80 palabras.

**B. Caso de análisis 💼** Una empresa de logística quiere predecir la demanda de los **próximos 14 días** a partir de los últimos 90. Un analista propone un seq2seq con LSTM; otra, 14 regresiones lineales (una por día del horizonte) sobre ventanas (Lección 11.1). ¿Qué ventajas tiene cada propuesta? ¿Con cuál empezarías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Para "Buenos días" el post-it alcanza de sobra. Para un párrafo de 80 palabras, el intérprete tendría que comprimir todos los matices en el mismo post-it pequeño: al recitar, lo del principio ya se borroneó y se inventa o se salta partes. Un intérprete real **vuelve a mirar sus notas** de cada frase; eso es la atención.

**B.** El **seq2seq** genera los 14 días como una secuencia coherente (cada día depende del anterior) y puede aprender patrones complejos, pero necesita muchos datos, es más difícil de entrenar y de explicar. Las **14 regresiones** son rápidas, interpretables y difíciles de vencer con pocos datos. Empezaría por las **líneas base** (estacional ingenua) y las regresiones sobre ventanas, con validación temporal; solo si el seq2seq les gana claramente en validación, valdría su complejidad. Es justo lo que harás en el proyecto del módulo.
</details>

### 9.3 🧩 Práctica integradora: una calculadora seq2seq ➕

Vas a entrenar un seq2seq que **suma**: recibe `"123+045"` como secuencia de símbolos y debe responder `"0168"`, dígito a dígito. Nadie le explica qué es sumar: solo ve ejemplos.

**Paso 1 ▶️ — Los datos.** Números de 0 a 999, escritos siempre con 3 dígitos. La respuesta tiene siempre 4 dígitos.

In [ ]:
MAS, INICIO_S, VOCAB_S = 10, 11, 12       # símbolos: 0–9, "+" y <inicio>

def digitos(v, k):
    return torch.stack([(v // 10 ** (k - 1 - i)) % 10 for i in range(k)], dim=1)

def lote_suma(n, generador, al_reves=False):
    a = torch.randint(0, 1000, (n,), generator=generador)
    b = torch.randint(0, 1000, (n,), generator=generador)
    x = torch.cat([digitos(a, 3), torch.full((n, 1), MAS), digitos(b, 3)], dim=1)
    y = digitos(a + b, 4)
    if al_reves:
        y = y.flip(1)                   # ✏️ (lo usarás en el Paso 4)
    return x, y

def mostrar(x, y):
    simbolo = "0123456789+"
    return "".join(simbolo[i] for i in x) + " = " + "".join(str(i) for i in y)

x, y = lote_suma(3, torch.Generator().manual_seed(0))
for a, b in zip(x.tolist(), y.tolist()):
    print(mostrar(a, b))

**Paso 2 ▶️ — Entrena.** El mismo seq2seq de la lección, adaptado a estos símbolos, con un contexto de 128 números. ▶️ ≈20–40 s.

In [ ]:
class Calculadora(nn.Module):
    def __init__(self, oculto=128):
        super().__init__()
        self.emb = nn.Embedding(VOCAB_S, 16)
        self.codificador = nn.GRU(16, oculto, batch_first=True)
        self.decodificador = nn.GRU(16, oculto, batch_first=True)
        self.salida = nn.Linear(oculto, 10)
    def forward(self, x, y):
        _, contexto = self.codificador(self.emb(x))
        entrada_dec = torch.cat([torch.full((len(y), 1), INICIO_S), y[:, :-1]], dim=1)
        salidas, _ = self.decodificador(self.emb(entrada_dec), contexto)
        return self.salida(salidas)
    @torch.no_grad()
    def responder(self, x, largo=4):
        _, h = self.codificador(self.emb(x))
        token, resultado = torch.full((len(x), 1), INICIO_S), []
        for _ in range(largo):
            salida, h = self.decodificador(self.emb(token), h)
            token = self.salida(salida).argmax(-1)
            resultado.append(token)
        return torch.cat(resultado, dim=1)

def entrenar_calculadora(al_reves=False, pasos=1000, semilla=0):
    torch.manual_seed(semilla); gen = torch.Generator().manual_seed(semilla)
    modelo = Calculadora()
    opt = torch.optim.Adam(modelo.parameters(), lr=0.005)
    inicio = time.time()
    for _ in range(pasos):
        x, y = lote_suma(128, gen, al_reves)
        perdida = F.cross_entropy(modelo(x, y).reshape(-1, 10), y.reshape(-1))
        opt.zero_grad(); perdida.backward()
        nn.utils.clip_grad_norm_(modelo.parameters(), 1.0)
        opt.step()
    x, y = lote_suma(1000, torch.Generator().manual_seed(99), al_reves)
    aciertos = modelo.responder(x) == y
    print(f"({time.time() - inicio:.0f} s) Sumas perfectas: {aciertos.all(1).float().mean():.0%}")
    print("Acierto por posición de la respuesta:", [f"{p:.0%}" for p in aciertos.float().mean(0).tolist()])
    return modelo

calculadora = entrenar_calculadora()

🔮 La respuesta se escribe como `millares, centenas, decenas, unidades`. ¿Qué posición crees que se le da **peor**?

✍️ *Mi apuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

Muy pocas sumas perfectas (≈5 %). Por posición: millares ≈100 %, centenas ≈90–95 %, decenas ≈30–40 % y **unidades ≈10 %** (¡como adivinar!). Es sorprendente: las **unidades** son lo más fácil para una persona (solo dependen de los dos últimos dígitos). Pero el modelo las escribe **al final**, después de generar los otros tres dígitos, cuando el post-it ya se gastó.
</details>

**Paso 3 ✋ — Piensa como en la primaria.** ¿En qué orden sumas tú a mano `123 + 045`? ¿Por qué ese orden es más fácil?

✍️ *Mi respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

De **derecha a izquierda**: primero las unidades (3 + 5 = 8), luego las decenas (2 + 4 = 6, más lo que "llevas"), etc. Cada dígito solo necesita los dos dígitos de su columna **y el acarreo** del paso anterior. Si escribes primero los millares, tienes que haber resuelto antes **toda** la cadena de acarreos de cabeza.
</details>

**Paso 4 ✏️ — Cambia el orden de la respuesta.** Entrena de nuevo pidiéndole la respuesta **al revés** (unidades primero). Solo cambia `al_reves=True`. ▶️ ≈20–40 s.

In [ ]:
calculadora_reves = entrenar_calculadora(al_reves=True)     # ✏️

<details><summary><b>👀 Ver respuesta</b></summary>

¡Salto enorme! Las sumas perfectas pasan de ≈5 % a cerca del **90 %**, con todas las posiciones por encima del 90 %. **Mismo modelo, mismos datos, mismo entrenamiento**: solo cambiamos **cómo se presenta la salida**. Con las unidades primero, el decodificador puede "llevar" el acarreo de un paso al siguiente, como tú en la primaria.
</details>

**Paso 5 ✋ — Lo que aprendiste.** Responde en dos o tres frases:
1. ¿Qué te dice este experimento sobre la importancia de **cómo representas** los datos de una secuencia?
2. ¿Cómo se relaciona con el cuello de botella y con el truco de Sutskever (Ejercicio 10)?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. En modelos de secuencias, **el orden en que presentas** la información puede importar tanto como la arquitectura. Es ingeniería de características (Lección 2.4) aplicada al tiempo: pon primero lo que se puede resolver con lo que ya sabes.
2. Con el orden "normal", el modelo tenía que resolver toda la suma **dentro del post-it** antes de escribir el primer dígito. Al invertir la salida, cada paso necesita poca información nueva y el decodificador la va acumulando: le quitamos presión al cuello de botella. Sutskever hizo lo mismo invirtiendo la entrada. Son trucos ingeniosos… que la **atención** volvió casi innecesarios (Lección 12.1).
</details>

---
## 📚 Para profundizar
- 📘 **Artículo:** Sutskever, Vinyals y Le (2014), *Sequence to Sequence Learning with Neural Networks* (el seq2seq original, con el truco de invertir la entrada).
- 📘 **Artículo:** Bahdanau, Cho y Bengio (2014), *Neural Machine Translation by Jointly Learning to Align and Translate* (la atención nace aquí).
- 📘 **Artículo visual:** Jay Alammar, *Visualizing A Neural Machine Translation Model (Mechanics of Seq2seq Models With Attention)* (inglés).
- 📺 **Tutorial:** PyTorch, *NLP From Scratch: Translation with a Sequence to Sequence Network and Attention*.

---
⬅️ **Anterior:** [11.4 LSTM y GRU: puertas para recordar y olvidar](11.4-lstm-y-gru.ipynb) · ➡️ **Siguiente:** [11.6 Proyecto del módulo: ¿Cuántas bicis mañana? 🚲](11.6-proyecto-del-modulo.md)